# Train the ABSA model

must have following OUTCOME(DIR):
- `two_stage_checkpoint.pt`
- `aspect_category_vocab.json`
- `thresholds.npy`
- `test_results.json`

Set `SMOKE_TEST = True` below for a fast pipeline check (a few minutes) instead
of a full run.

In [ ]:
import json
import time
import tracemalloc
from collections import Counter
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

MODEL_NAME = "microsoft/deberta-v3-base"
POLARITIES = ["positive", "negative", "neutral"]
POL2IDX = {p: i for i, p in enumerate(POLARITIES)}
SENTENCE_MAX_LEN = 128
REVIEW_MAX_LEN = 384

TRAIN_URL = "https://huggingface.co/datasets/alexcadillon/SemEval2016Task5/resolve/refs%2Fconvert%2Fparquet/laptops/train/0000.parquet"
TEST_URL = "https://huggingface.co/datasets/alexcadillon/SemEval2016Task5/resolve/refs%2Fconvert%2Fparquet/laptops/test/0000.parquet"

## Mount Google Drive (Colab only)

Saves every generated file (checkpoints, vocab, thresholds, results) straight to Drive as it's produced

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/absa_training_output"
    import os
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    IN_COLAB = True
    print(f"Running in Colab — outputs will be saved to: {DRIVE_OUTPUT_DIR}")
except ImportError:
    DRIVE_OUTPUT_DIR = "."
    IN_COLAB = False
    print("Not running in Colab — outputs will be saved to the current directory.")

## Run configuration


In [ ]:
EPOCHS_STAGE1 = 6
EPOCHS_STAGE2 = 10
LR_STAGE1 = 2e-5
LR_STAGE2 = 8e-6
FREEZE_LAYERS = 4
OUTPUT_DIR = DRIVE_OUTPUT_DIR  # set above: Drive path in Colab, "." otherwise
SMOKE_TEST = False  # subsample data and run 1 epoch per stage — validate the pipeline in minutes

# Threshold/precision tuning — see the low-precision analysis: these guard
# against per-category thresholds overfitting on low-support validation data.
CATEGORY_POS_WEIGHT_CAP = 8.0     # was 20.0 — less aggressive upweighting of rare categories
THRESHOLD_FLOOR = 0.35            # a tuned threshold can never go below this
MIN_THRESHOLD_TUNING_SUPPORT = 5  # categories with fewer than this many val positives keep threshold=0.5

if SMOKE_TEST:
    EPOCHS_STAGE1 = 1
    EPOCHS_STAGE2 = 1

## Data loading / flattening

In [ ]:
def flat_category(cat):
    if cat is None:
        return "NULL#NULL"
    e = cat.get("entity") or "NULL"
    a = cat.get("attribute") or "NULL"
    return f"{e}#{a}"


def build_review_level(split):
    rows = []
    for row in split:
        text = " ".join(s["text"] for s in row["sentences"])
        opinions = [{"category": flat_category(o["category"]), "polarity": o["polarity"]}
                    for o in row["opinions"]]
        rows.append({"reviewId": row["reviewId"], "text": text, "opinions": opinions})
    return rows


def build_sentence_level(split):
    rows = []
    for row in split:
        for s in row["sentences"]:
            opinions = [{"category": flat_category(o["category"]), "polarity": o["polarity"]}
                        for o in s["opinions"]]
            rows.append({"reviewId": row["reviewId"], "sentenceId": s["sentenceId"],
                         "text": s["text"], "opinions": opinions})
    return rows

## Dataset / model

In [ ]:
class ABSADataset(Dataset):
    def __init__(self, records, tokenizer, cat2idx, max_length):
        self.records = records
        self.tokenizer = tokenizer
        self.cat2idx = cat2idx
        self.num_categories = len(cat2idx)
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        cat_labels = np.zeros(self.num_categories, dtype=np.float32)
        pol_labels = np.full(self.num_categories, -1, dtype=np.int64)
        for o in r["opinions"]:
            if o["category"] not in self.cat2idx:
                continue
            ci = self.cat2idx[o["category"]]
            cat_labels[ci] = 1.0
            pol_labels[ci] = POL2IDX.get(o["polarity"], -1)

        enc = self.tokenizer(r["text"], truncation=True, padding="max_length",
                              max_length=self.max_length, return_tensors="pt")
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "category_labels": torch.tensor(cat_labels),
            "polarity_labels": torch.tensor(pol_labels),
        }

In [ ]:
class JointABSAModel(nn.Module):
    def __init__(self, model_name, num_categories, num_polarities=3, dropout=0.2):
        super().__init__()
        # torch_dtype=float32 avoids a Half/Float dtype mismatch some Hub
        # checkpoints trigger against these newly-initialized float32 heads.
        self.encoder = AutoModel.from_pretrained(model_name, torch_dtype=torch.float32)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.category_head = nn.Linear(hidden, num_categories)
        self.polarity_head = nn.Linear(hidden, num_categories * num_polarities)
        self.num_categories = num_categories
        self.num_polarities = num_polarities

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(out.last_hidden_state[:, 0])
        cat_logits = self.category_head(pooled)
        pol_logits = self.polarity_head(pooled).view(-1, self.num_categories, self.num_polarities)
        return cat_logits, pol_logits

In [ ]:
def freeze_bottom_layers(model, n):
    """Freeze the bottom N transformer blocks before stage 2 — keeps what
    stage 1 learned, lets only the top layers + heads adapt."""
    if n <= 0:
        return
    try:
        layers = model.encoder.encoder.layer
    except AttributeError:
        print("Could not locate encoder layers to freeze — skipping.")
        return
    for layer in layers[:n]:
        for p in layer.parameters():
            p.requires_grad = False
    print(f"Froze bottom {n} encoder layers.")

## Loss weighting / evaluation helpers

In [ ]:
def compute_category_pos_weight(records, cat2idx, cap=CATEGORY_POS_WEIGHT_CAP):
    counts = np.zeros(len(cat2idx))
    for r in records:
        seen = {o["category"] for o in r["opinions"] if o["category"] in cat2idx}
        for c in seen:
            counts[cat2idx[c]] += 1
    n = len(records)
    pos = np.clip(counts, 1, None)
    return torch.tensor(np.clip((n - pos) / pos, 0, cap), dtype=torch.float32)


def compute_polarity_class_weight(records):
    counts = np.zeros(len(POLARITIES))
    for r in records:
        for o in r["opinions"]:
            if o["polarity"] in POL2IDX:
                counts[POL2IDX[o["polarity"]]] += 1
    counts = np.clip(counts, 1, None)
    return torch.tensor(counts.sum() / (len(POLARITIES) * counts), dtype=torch.float32)


def tune_thresholds(cat_probs, cat_labels, num_categories, grid=np.arange(0.1, 0.9, 0.05),
                     min_threshold=THRESHOLD_FLOOR, min_support=MIN_THRESHOLD_TUNING_SUPPORT):
    """Grid-search a per-category sigmoid cutoff.

    Two guards against the low-precision failure mode where a rare category
    gets a degenerate threshold from overfitting a handful of validation
    examples:
    - min_support: categories with fewer than this many positives in
      `cat_labels` skip tuning and keep the default 0.5 threshold instead.
    - min_threshold: the tuned value is never allowed to go below this floor,
      so a category can't end up firing on almost everything.
    """
    thresholds = np.full(num_categories, 0.5)
    for i in range(num_categories):
        if cat_labels[:, i].sum() < min_support:
            continue
        best_f1, best_t = 0, 0.5
        for t in grid:
            preds = (cat_probs[:, i] >= t).astype(int)
            f1 = f1_score(cat_labels[:, i], preds, zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        thresholds[i] = max(best_t, min_threshold)
    return thresholds

In [ ]:
def exact_match_tuple_f1(records, cat_probs, pol_logits, category_vocab, thresholds):
    tp, fp, fn = 0, 0, 0
    pol_preds_all = pol_logits.argmax(-1)
    for idx, r in enumerate(records):
        gold = {(o["category"], o["polarity"]) for o in r["opinions"]}
        pred = set()
        for ci, cat in enumerate(category_vocab):
            if cat_probs[idx, ci] >= thresholds[ci]:
                pred.add((cat, POLARITIES[pol_preds_all[idx, ci]]))
        tp += len(gold & pred)
        fp += len(pred - gold)
        fn += len(gold - pred)
    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    return {"precision": precision, "recall": recall, "f1": f1, "tp": tp, "fp": fp, "fn": fn}

In [ ]:
def run_epoch(model, loader, cat_loss_fn, pol_loss_fn, device, optimizer=None, scheduler=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss = 0
    all_cat_probs, all_cat_labels, all_pol_logits = [], [], []
    pbar = tqdm(loader, desc="train" if training else "eval", leave=False)
    for batch in pbar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        cat_labels = batch["category_labels"].to(device)
        pol_labels = batch["polarity_labels"].to(device)
        with torch.set_grad_enabled(training):
            cat_logits, pol_logits = model(input_ids, attention_mask)
            cat_loss = cat_loss_fn(cat_logits, cat_labels)
            pol_loss = pol_loss_fn(pol_logits.reshape(-1, len(POLARITIES)), pol_labels.reshape(-1))
            loss = cat_loss + pol_loss
            if training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")
        all_cat_probs.append(torch.sigmoid(cat_logits).detach().cpu().numpy())
        all_cat_labels.append(cat_labels.cpu().numpy())
        all_pol_logits.append(pol_logits.detach().cpu().numpy())
    return (total_loss / len(loader), np.concatenate(all_cat_probs),
            np.concatenate(all_cat_labels), np.concatenate(all_pol_logits))

In [ ]:
def train_loop(model, train_loader, val_loader, val_records, category_vocab, device,
               epochs, lr, checkpoint_path, patience=4):
    pos_weight = compute_category_pos_weight(train_loader.dataset.records, train_loader.dataset.cat2idx).to(device)
    cat_loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    pol_weight = compute_polarity_class_weight(train_loader.dataset.records).to(device)
    pol_loss_fn = nn.CrossEntropyLoss(weight=pol_weight, ignore_index=-1)

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=0.05)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)

    best_f1, best_thresholds, patience_counter = -1, None, 0
    history = {"train_loss": [], "val_loss": [], "val_f1": [], "val_precision": [], "val_recall": []}

    for epoch in range(epochs):
        train_loss, _, _, _ = run_epoch(model, train_loader, cat_loss_fn, pol_loss_fn, device, optimizer, scheduler)
        val_loss, val_cat_probs, val_cat_labels, val_pol_logits = run_epoch(model, val_loader, cat_loss_fn, pol_loss_fn, device)
        thresholds = tune_thresholds(val_cat_probs, val_cat_labels, len(category_vocab))
        val_metrics = exact_match_tuple_f1(val_records, val_cat_probs, val_pol_logits, category_vocab, thresholds)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_f1"].append(val_metrics["f1"])
        history["val_precision"].append(val_metrics["precision"])
        history["val_recall"].append(val_metrics["recall"])

        print(f"  epoch {epoch+1}: train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
              f"val_f1={val_metrics['f1']:.4f} (P={val_metrics['precision']:.3f} R={val_metrics['recall']:.3f})")

        if val_metrics["f1"] > best_f1:
            best_f1, best_thresholds, patience_counter = val_metrics["f1"], thresholds, 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("  early stopping.")
                break

    model.load_state_dict(torch.load(checkpoint_path))
    return model, best_thresholds, best_f1, history

In [ ]:
def plot_training_curves(history, stage_name):
    epochs_range = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    axes[0].plot(epochs_range, history["train_loss"], marker="o", label="train loss")
    axes[0].plot(epochs_range, history["val_loss"], marker="o", label="val loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title(f"{stage_name}: loss per epoch")
    axes[0].legend()

    axes[1].plot(epochs_range, history["val_f1"], marker="o", color="green", label="val F1")
    axes[1].plot(epochs_range, history["val_precision"], marker="o", color="steelblue", linestyle="--", label="val precision")
    axes[1].plot(epochs_range, history["val_recall"], marker="o", color="darkorange", linestyle="--", label="val recall")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Score")
    axes[1].set_ylim(0, 1)
    axes[1].set_title(f"{stage_name}: validation metrics per epoch")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

## Setup: device + dataset loading

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cpu":
    print("WARNING: no GPU detected — training will be slow. "
          "On Colab: Runtime > Change runtime type > GPU.")

In [ ]:
print("Loading dataset...")
ds = load_dataset("parquet", data_files={"train": TRAIN_URL, "test": TEST_URL})

review_train_all = build_review_level(ds["train"])
review_test = build_review_level(ds["test"])
sentence_train_all = build_sentence_level(ds["train"])

## Build unified category vocab

Unified across both levels so stage-1 and stage-2 label indices line up exactly.

In [ ]:
all_categories = {o["category"] for r in review_train_all for o in r["opinions"]}
all_categories |= {o["category"] for r in sentence_train_all for o in r["opinions"]}
category_vocab = sorted(all_categories)
cat2idx = {c: i for i, c in enumerate(category_vocab)}
with open(f"{OUTPUT_DIR}/aspect_category_vocab.json", "w") as f:
    json.dump(category_vocab, f, indent=2)
print(f"{len(category_vocab)} unified categories")

### Chart 1 — aspect category frequency (class imbalance)

In [ ]:
cat_counts = Counter()
for r in review_train_all:
    for o in r["opinions"]:
        cat_counts[o["category"]] += 1

top_cats = cat_counts.most_common(20)
labels, counts = zip(*top_cats)

plt.figure(figsize=(10, 6))
sns.barplot(x=list(counts), y=list(labels), color="steelblue")
plt.xlabel("Occurrences (review-level train)")
plt.title(f"Top {len(top_cats)} of {len(category_vocab)} aspect categories by frequency")
plt.tight_layout()
plt.show()

### Chart 2 — polarity label distribution

In [ ]:
pol_counts = Counter()
for r in review_train_all:
    for o in r["opinions"]:
        if o["polarity"] in POL2IDX:
            pol_counts[o["polarity"]] += 1

plt.figure(figsize=(5, 5))
colors = {"positive": "#4CAF50", "negative": "#E53935", "neutral": "#9E9E9E"}
plt.pie([pol_counts[p] for p in POLARITIES], labels=POLARITIES, autopct="%1.1f%%",
        colors=[colors[p] for p in POLARITIES], startangle=90)
plt.title("Polarity distribution (review-level train)")
plt.tight_layout()
plt.show()

### Chart 3 — text length distribution

Sanity-checks `SENTENCE_MAX_LEN` / `REVIEW_MAX_LEN`

In [ ]:
sent_lengths = [len(r["text"].split()) for r in sentence_train_all]
review_lengths = [len(r["text"].split()) for r in review_train_all]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(sent_lengths, bins=30, color="steelblue", edgecolor="white")
axes[0].axvline(SENTENCE_MAX_LEN, color="crimson", linestyle="--", label=f"max_len={SENTENCE_MAX_LEN} tokens")
axes[0].set_title("Sentence-level text length (words)")
axes[0].set_xlabel("Word count")
axes[0].legend()

axes[1].hist(review_lengths, bins=30, color="darkorange", edgecolor="white")
axes[1].axvline(REVIEW_MAX_LEN, color="crimson", linestyle="--", label=f"max_len={REVIEW_MAX_LEN} tokens")
axes[1].set_title("Review-level text length (words)")
axes[1].set_xlabel("Word count")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
sent_has_op = [1 if len(r["opinions"]) > 0 else 0 for r in sentence_train_all]
sent_train, sent_val = train_test_split(sentence_train_all, test_size=0.1, random_state=42, stratify=sent_has_op)

rev_has_op = [1 if len(r["opinions"]) > 0 else 0 for r in review_train_all]
review_train, review_val = train_test_split(review_train_all, test_size=0.15, random_state=42, stratify=rev_has_op)

if SMOKE_TEST:
    sent_train = sent_train[:200]
    review_train = review_train[:60]
    review_test = review_test[:20]
    print("Smoke test mode: subsampled to 200 sentence-train / 60 review-train / 20 test")

### Chart 4 — split sizes


In [ ]:
split_names = ["sent_train", "sent_val", "review_train", "review_val", "review_test"]
split_sizes = [len(sent_train), len(sent_val), len(review_train), len(review_val), len(review_test)]

plt.figure(figsize=(8, 4.5))
sns.barplot(x=split_names, y=split_sizes, palette="Blues_d")
for i, v in enumerate(split_sizes):
    plt.text(i, v, str(v), ha="center", va="bottom")
plt.ylabel("Records")
plt.title("Dataset split sizes")
plt.tight_layout()
plt.show()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = JointABSAModel(MODEL_NAME, num_categories=len(category_vocab)).to(device)

## Stage 1: pretrain on sentence-level pool

In [ ]:
sent_train_ds = ABSADataset(sent_train, tokenizer, cat2idx, SENTENCE_MAX_LEN)
sent_val_ds = ABSADataset(sent_val, tokenizer, cat2idx, SENTENCE_MAX_LEN)
sent_train_loader = DataLoader(sent_train_ds, batch_size=16, shuffle=True)
sent_val_loader = DataLoader(sent_val_ds, batch_size=32)

model, _, stage1_f1, stage1_history = train_loop(
    model, sent_train_loader, sent_val_loader, sent_val, category_vocab, device,
    epochs=EPOCHS_STAGE1, lr=LR_STAGE1,
    checkpoint_path=f"{OUTPUT_DIR}/stage1_checkpoint.pt",
)
print(f"Stage 1 best val F1 (sentence-level): {stage1_f1:.4f}")
freeze_bottom_layers(model, FREEZE_LAYERS)

### Chart 5 — stage 1 training curves

In [ ]:
plot_training_curves(stage1_history, "Stage 1 (sentence-level)")

## Stage 2: fine-tune on document-level (review) pool

In [ ]:
rev_train_ds = ABSADataset(review_train, tokenizer, cat2idx, REVIEW_MAX_LEN)
rev_val_ds = ABSADataset(review_val, tokenizer, cat2idx, REVIEW_MAX_LEN)
rev_train_loader = DataLoader(rev_train_ds, batch_size=8, shuffle=True)
rev_val_loader = DataLoader(rev_val_ds, batch_size=16)

model, thresholds, stage2_f1, stage2_history = train_loop(
    model, rev_train_loader, rev_val_loader, review_val, category_vocab, device,
    epochs=EPOCHS_STAGE2, lr=LR_STAGE2,
    checkpoint_path=f"{OUTPUT_DIR}/two_stage_checkpoint.pt",
)
print(f"Stage 2 best val F1 (document-level): {stage2_f1:.4f}")
np.save(f"{OUTPUT_DIR}/thresholds.npy", thresholds)
print("Saved thresholds.npy")

### Chart 6 — stage 2 training curves

In [ ]:
plot_training_curves(stage2_history, "Stage 2 (review-level fine-tune)")

### Chart 7 — per-category decision thresholds


In [ ]:
plt.figure(figsize=(10, 6))
sorted_idx = np.argsort(thresholds)
plt.barh(range(len(thresholds)), thresholds[sorted_idx], color="mediumpurple")
plt.axvline(0.5, color="gray", linestyle="--", label="default 0.5")
plt.yticks([])
plt.xlabel("Tuned threshold")
plt.title(f"Tuned decision thresholds across {len(thresholds)} categories (sorted)")
plt.legend()
plt.tight_layout()
plt.show()

## Final test evaluation

In [ ]:
test_ds = ABSADataset(review_test, tokenizer, cat2idx, REVIEW_MAX_LEN)
test_loader = DataLoader(test_ds, batch_size=16)

pos_weight = compute_category_pos_weight(review_train, cat2idx).to(device)
cat_loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
pol_weight = compute_polarity_class_weight(review_train).to(device)
pol_loss_fn = nn.CrossEntropyLoss(weight=pol_weight, ignore_index=-1)

t0 = time.perf_counter()
tracemalloc.start()
_, test_cat_probs, test_cat_labels, test_pol_logits = run_epoch(model, test_loader, cat_loss_fn, pol_loss_fn, device)
_, peak_mem = tracemalloc.get_traced_memory()
tracemalloc.stop()
test_time = time.perf_counter() - t0

test_metrics = exact_match_tuple_f1(review_test, test_cat_probs, test_pol_logits, category_vocab, thresholds)
test_metrics["ms_per_review"] = test_time / len(review_test) * 1000
test_metrics["peak_memory_mb"] = peak_mem / 1e6

print(json.dumps(test_metrics, indent=2))
with open(f"{OUTPUT_DIR}/test_results.json", "w") as f:
    json.dump(test_metrics, f, indent=2)
print("Saved test_results.json")

### Chart 8 — final precision / recall / F1

Exact-match (category, polarity) tuple metrics on the held-out test set.

In [ ]:
plt.figure(figsize=(5, 4.5))
metric_names = ["precision", "recall", "f1"]
metric_values = [test_metrics[m] for m in metric_names]
bars = plt.bar(metric_names, metric_values, color=["steelblue", "darkorange", "seagreen"])
plt.ylim(0, 1)
for b, v in zip(bars, metric_values):
    plt.text(b.get_x() + b.get_width() / 2, v, f"{v:.3f}", ha="center", va="bottom")
plt.title("Final test set metrics")
plt.tight_layout()
plt.show()

### Chart 9 — polarity confusion matrix


In [ ]:
test_pol_preds = test_pol_logits.argmax(-1)
y_true, y_pred = [], []
for idx, r in enumerate(review_test):
    gold_by_cat = {o["category"]: o["polarity"] for o in r["opinions"]}
    for ci, cat in enumerate(category_vocab):
        if test_cat_probs[idx, ci] >= thresholds[ci] and cat in gold_by_cat and gold_by_cat[cat] in POL2IDX:
            y_true.append(POL2IDX[gold_by_cat[cat]])
            y_pred.append(test_pol_preds[idx, ci])

if y_true:
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(POLARITIES))))
    plt.figure(figsize=(5, 4.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=POLARITIES, yticklabels=POLARITIES)
    plt.xlabel("Predicted polarity")
    plt.ylabel("Gold polarity")
    plt.title(f"Polarity confusion matrix (n={len(y_true)} matched aspects)")
    plt.tight_layout()
    plt.show()
else:
    print("No matched (true-positive) aspect predictions to build a confusion matrix from.")

### Chart 10 — per-category F1 (aspect detection only)


In [ ]:
test_cat_preds = (test_cat_probs >= thresholds).astype(int)
per_cat_f1 = [f1_score(test_cat_labels[:, i], test_cat_preds[:, i], zero_division=0)
              for i in range(len(category_vocab))]
order = np.argsort(per_cat_f1)
worst = order[:10]
best = order[-10:]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].barh([category_vocab[i] for i in worst], [per_cat_f1[i] for i in worst], color="indianred")
axes[0].set_xlim(0, 1)
axes[0].set_title("10 lowest-F1 categories")

axes[1].barh([category_vocab[i] for i in best], [per_cat_f1[i] for i in best], color="mediumseagreen")
axes[1].set_xlim(0, 1)
axes[1].set_title("10 highest-F1 categories")

plt.tight_layout()
plt.show()